In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import platform
import seaborn as sns
sns.set_theme(style="darkgrid")
import koreanize_matplotlib
from io import StringIO

In [ ]:
df = pd.read_json('./data/문장교정기록.json')

# 2. MongoDB BSON 타입($oid, $date 등)을 일반 값으로 변환하는 함수 정의
def normalize_mongo_cell(x):
    # 1) 딕셔너리인 경우: 특정 키($oid, $date)가 있는지 확인 후 값 추출
    if isinstance(x, dict):
        if '$oid' in x:
            return x['$oid']
        elif '$date' in x:
            return x['$date']
        elif '$numberLong' in x:
            return int(x['$numberLong'])
        # 다른 키가 들어있는 일반 딕셔너리라면 그대로 반환 (또는 필요시 로직 추가)
        return x
        
    # 2) 리스트인 경우: 리스트 안의 요소들에 대해 재귀적으로 함수 적용
    # 예: models: [{"$oid": "A"}, {"$oid": "B"}] -> ["A", "B"]
    elif isinstance(x, list):
        return [normalize_mongo_cell(item) for item in x]
        
    # 3) 그 외(숫자, 문자열, None 등)는 그대로 반환
    return x

# 3. 모든 컬럼을 순회하며 변환 적용
# apply 함수는 시리즈(컬럼)의 각 데이터에 normalize_mongo_cell 함수를 실행합니다.
for col in df.columns:
    df[col] = df[col].apply(normalize_mongo_cell)

# 4. 날짜 형식 문자열을 실제 datetime 객체로 변환 (선택 사항)
# 'created_at'이나 'updated_at' 같은 이름이 포함된 컬럼이 있다면 변환 시도
for col in df.columns:
    if 'date' in col or '_at' in col:
        # 에러 발생 시(날짜 아님) 무시하고 원본 유지(coerce)
        df[col] = pd.to_datetime(df[col], errors='ignore') 

# 5. properties 컬럼 펼치기 (사용자가 제시한 코드 로직 유지)
# properties가 있고 딕셔너리 형태라면 펼쳐서 병합합니다.
if 'properties' in df.columns:
    # 딕셔너리가 아닌 데이터(None 등)가 섞여있을 때 에러 방지
    valid_props = df['properties'].apply(lambda x: x if isinstance(x, dict) else {})
    props_df3 = pd.json_normalize(valid_props)
    
    # 중복되는 컬럼(_id, distinct_id, user_id 등) 제거
    cols_to_drop = [c for c in ['distinct_id', 'user_id', '_id'] if c in props_df3.columns]
    props_df3 = props_df3.drop(cols_to_drop, axis=1)
    
    # 원본 df에서 properties 제거 후 펼친 데이터 병합
    df = pd.concat([df.drop('properties', axis=1), props_df3], axis=1)

In [ ]:
df

In [ ]:
df[df['selected_index'].notnull()]

In [ ]:
df_clean = df.dropna(subset=['selected_index']).copy()
df_clean['selected_index'] = df_clean['selected_index'].astype(int)

# 4. 최종 교정문(Label) 추출 함수
def get_final_pair(row):
    outputs = row['output_sentences']
    idx = row['selected_index']
    
    # 데이터 무결성 체크 (리스트가 맞는지, 인덱스가 범위 내에 있는지)
    if not isinstance(outputs, list) or len(outputs) == 0:
        return None
    
    if 0 <= idx < len(outputs):
        return outputs[idx]
    else:
        return None  # 인덱스가 범위를 벗어난 이상 데이터

# 함수 적용
df_clean['final_corrected'] = df_clean.apply(get_final_pair, axis=1)

# 5. 데이터 클렌징 (추출 과정에서 None이 되었거나 빈 문장 제거)
df_clean = df_clean.dropna(subset=['final_corrected'])
df_clean = df_clean[df_clean['input_sentence'].str.strip() != ""]
df_clean = df_clean[df_clean['final_corrected'].str.strip() != ""]

# 결과 확인
print("-" * 30)
print(f"전처리 후 학습 가능한 데이터 개수: {len(df_clean)}")
print("-" * 30)
print(df_clean[['input_sentence', 'final_corrected','intensity','field']].head())

In [ ]:
output = df_clean[['input_sentence', 'final_corrected','intensity','field']]

In [ ]:
output.to_csv('./data/munch_notlabeled.csv', index=False)

In [ ]:
df_res = pd.read_csv('./data/munch_notlabeled.csv')

In [ ]:
df_res.head()

In [ ]:
df_res['field'].value_counts()